In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [7]:
import pandas as pd
from langchain_core.documents import Document

In [ ]:
class DataIngestion:
    def __init__(self,csv_path):
        self.csv_path=csv_path
    def load_dataset(self):
        df=pd.read_csv(self.csv_path)
        print(f"Loaded{len(df)} tourist places.")
        return df
    def create_documents(self,df):
        documents=[]
        for _, row in df.iterrows():

            content = f"""
            Place: {row['Name']}
            City: {row['City']}
            State: {row['State']}
            Type: {row['Type']}
            Significance: {row['Significance']}
            Best Time to Visit: {row['Best Time to visit']}
            Visit Duration: {row['time needed to visit in hrs']} hours
            Google Rating: {row['Google review rating']}
            Entrance Fee: {row['Entrance Fee in INR']} INR
            """
            doc = Document(page_content=content, metadata={
                    "name": row["Name"],
                    "city": row["City"],
                    "state": row["State"],
                    "type": row["Type"],
                    "rating": row["Google review rating"],
                    "duration": row["time needed to visit in hrs"],
                    "fee": row["Entrance Fee in INR"],
                    "best_time": row["Best Time to visit"]
                })
            documents.append(doc)
        
        print(f"Created {len(documents)} LangChain documents.")

        return documents

In [14]:
from langchain_huggingface import HuggingFaceEmbeddings
class EmbeddingModel:
    def __init__(self,model_name="sentence-transformers/all-MiniLM-L6-v2"):
        self.embedding_model=HuggingFaceEmbeddings(model_name=model_name)
    def creating_embedding(self,documents):
        text=[doc.page_content for doc in documents]
        embeddings=self.embedding_model.embed_documents(text)
        return embeddings
    def embed_query(self,query):
        return self.embedding_model.embed_query()

In [15]:
import os
import uuid
import chromadb


In [18]:
class VectorStore:
    def __init__(self,collection_name="tourist_places",presist_directory="./vector_db"):
        os.makedirs(presist_directory,exist_ok=True)
        self.client=chromadb.PersistentClient(path=presist_directory)
        self.collection=self.client.get_or_create_collection(name=collection_name)
    def add_documents(self,documents,embedding):
        ids=[str(uuid.uuid4()) for _ in documents]
        texts=[doc.page_content for doc in documents]
        metadatas=[doc.metadata for doc in documents]
        self.collection.add(ids=ids,documents=texts,embeddings=embedding,metadatas=metadatas)
    def similarity_search(self, query_embedding, top_k=5):

        results = self.collection.query(
            query_embeddings=[query_embedding],
            n_results=top_k
        )

        return results

In [19]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq


class RAGPipeline:

    def __init__(self, vector_store, embedding_model, model_name="llama-3.3-70b-versatile"):

        self.vector_store = vector_store
        self.embedding_model = embedding_model

        self.llm = ChatGroq(
            model=model_name,
            temperature=0
        )

        self.prompt = ChatPromptTemplate.from_template("""
You are ItinerAI, an intelligent AI Travel Planner.

Your task is to generate a personalized travel itinerary.

Instructions:
1. Use ONLY the retrieved tourist place information.
2. Do NOT invent tourist places.
3. Create a day-wise itinerary.
4. Mention:
   • Place Name
   • City
   • Visit Duration
   • Why visit
5. If the retrieved information is insufficient, clearly say so.

Retrieved Tourist Places:
{context}

User Request:
{question}

Generate a detailed travel itinerary.
""")

    def ask(self, query, top_k=5):

        # Step 1 : Convert user query into embedding
        query_embedding = self.embedding_model.embed_query(query)

        # Step 2 : Retrieve similar tourist places
        results = self.vector_store.similarity_search(
            query_embedding=query_embedding,
            top_k=top_k
        )

        # Step 3 : Build context
        documents = results["documents"][0]
        context = "\n\n".join(documents)

        # Step 4 : Create prompt
        formatted_prompt = self.prompt.format(
            context=context,
            question=query
        )

        # Step 5 : Generate answer
        response = self.llm.invoke(formatted_prompt)

        return {
            "answer": response.content,
            "documents": results["documents"][0],
            "metadata": results["metadatas"][0]
        }

In [20]:
class RecommendationEngine:

    def __init__(self):
        pass

    def recommend(
        self,
        places,
        city=None,
        budget=None,
        interest=None,
        days=1,
        places_per_day=3
    ):

        recommended = places

        # Filter by City
        if city:
            recommended = [
                place for place in recommended
                if place.get("city", "").lower() == city.lower()
            ]

        # Filter by Interest (Type)
        if interest:
            recommended = [
                place for place in recommended
                if interest.lower() in place.get("type", "").lower()
            ]

        # Filter by Budget (Entrance Fee)
        if budget is not None:
            recommended = [
                place for place in recommended
                if float(place.get("fee", 0)) <= budget
            ]

        # Sort by Rating (Highest First)
        recommended.sort(
            key=lambda x: float(x.get("rating", 0)),
            reverse=True
        )

        # Remove places with invalid duration
        recommended = [
            place for place in recommended
            if place.get("duration") not in [None, "", "nan"]
        ]

        # Select places based on trip duration
        max_places = days * places_per_day
        recommended = recommended[:max_places]

        return recommended


if __name__ == "__main__":

    # Sample metadata returned from your RAG Pipeline

    places = [
        {
            "name": "Amber Fort",
            "city": "Jaipur",
            "type": "Historical",
            "rating": 4.8,
            "fee": 200,
            "duration": 3
        },
        {
            "name": "Hawa Mahal",
            "city": "Jaipur",
            "type": "Historical",
            "rating": 4.7,
            "fee": 50,
            "duration": 2
        },
        {
            "name": "Jal Mahal",
            "city": "Jaipur",
            "type": "Lake",
            "rating": 4.5,
            "fee": 0,
            "duration": 1
        },
        {
            "name": "City Palace",
            "city": "Jaipur",
            "type": "Historical",
            "rating": 4.6,
            "fee": 300,
            "duration": 2
        }
    ]

    recommender = RecommendationEngine()

    recommendations = recommender.recommend(
        places=places,
        city="Jaipur",
        budget=500,
        interest="Historical",
        days=2
    )

    print("\nRecommended Places:\n")

    for i, place in enumerate(recommendations, start=1):
        print(f"{i}. {place['name']}")
        print(f"   City      : {place['city']}")
        print(f"   Type      : {place['type']}")
        print(f"   Rating    : {place['rating']}")
        print(f"   Fee       : ₹{place['fee']}")
        print(f"   Duration  : {place['duration']} hours")
        print()


Recommended Places:

1. Amber Fort
   City      : Jaipur
   Type      : Historical
   Rating    : 4.8
   Fee       : ₹200
   Duration  : 3 hours

2. Hawa Mahal
   City      : Jaipur
   Type      : Historical
   Rating    : 4.7
   Fee       : ₹50
   Duration  : 2 hours

3. City Palace
   City      : Jaipur
   Type      : Historical
   Rating    : 4.6
   Fee       : ₹300
   Duration  : 2 hours

